In [7]:
# ============================================================
# 20 NEWSGROUPS - TF-IDF & COSINE SIMILARITY
#
# ============================================================

import os
import re
import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


# ============================================================
# DATASET PATH
# ============================================================

DATA_PATH = "/kaggle/input/datasets/crawford/20-newsgroups"

print("=" * 70)
print("20 NEWSGROUPS DATASET")
print("=" * 70)


# ============================================================
# LOAD LIST.CSV TO UNDERSTAND DATASET
# ============================================================

list_path = os.path.join(DATA_PATH, "list.csv")

if os.path.exists(list_path):

    list_df = pd.read_csv(list_path)

    print("\nlist.csv columns:")
    print(list_df.columns.tolist())

    print("\nFirst rows:")
    print(list_df.head())


# ============================================================
# READ EACH .TXT FILE
# ============================================================
# Each TXT file represents one newsgroup.
# Documents inside the file are separated by blank lines
# containing the original 20 Newsgroups posts.
# ============================================================

txt_files = [
    f for f in os.listdir(DATA_PATH)
    if f.endswith(".txt")
]

print("\nNumber of category files:", len(txt_files))

print("\nAvailable categories:")
for f in txt_files:
    print(f.replace(".txt", ""))


# ============================================================
# PARSE DOCUMENTS
# ============================================================

documents = []
categories = []


def split_documents(text):

    # Normalize line endings
    text = text.replace("\r\n", "\n")

    # Try splitting documents using multiple blank lines
    docs = re.split(r"\n\s*\n\s*\n+", text)

    # If that produces very few documents,
    # try two or more blank lines
    if len(docs) < 100:
        docs = re.split(r"\n\s*\n+", text)

    docs = [
        doc.strip()
        for doc in docs
        if len(doc.strip()) > 50
    ]

    return docs


for file in txt_files:

    file_path = os.path.join(DATA_PATH, file)

    category = file.replace(".txt", "")

    with open(
        file_path,
        "r",
        encoding="latin-1",
        errors="ignore"
    ) as f:

        content = f.read()

    docs = split_documents(content)

    print(
        f"{category:35s} -> {len(docs)} documents"
    )

    for doc in docs:

        documents.append(doc)
        categories.append(category)


df = pd.DataFrame({
    "text": documents,
    "class": categories
})


print("\nTotal documents detected:", len(df))

print("\nDocuments per category:")
print(df["class"].value_counts())


# ============================================================
# SELECT THREE CATEGORIES
# ============================================================

np.random.seed(42)

class_counts = df["class"].value_counts()

valid_classes = class_counts[
    class_counts >= 100
].index.tolist()

if len(valid_classes) < 3:

    raise ValueError(
        "The TXT files were not split into individual documents correctly."
    )


# Randomly select three categories
selected_classes = np.random.choice(
    valid_classes,
    size=3,
    replace=False
)

print("\n" + "=" * 70)
print("SELECTED CATEGORIES")
print("=" * 70)

print(selected_classes)


# ============================================================
# SELECT 100 DOCUMENTS FROM EACH CATEGORY
# ============================================================

selected_data = []

for category in selected_classes:

    category_df = df[
        df["class"] == category
    ]

    sample = category_df.sample(
        n=100,
        random_state=42
    )

    selected_data.append(sample)


df_selected = pd.concat(
    selected_data,
    ignore_index=True
)

# Shuffle all 300 documents
df_selected = df_selected.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print("\nFinal dataset size:", len(df_selected))

print("\nDocuments per selected class:")
print(
    df_selected["class"].value_counts()
)


# ============================================================
# 1. PREPROCESSING
# ============================================================

stop_words = set(
    stopwords.words("english")
)

lemmatizer = WordNetLemmatizer()


def preprocess(text):

    # Lowercase
    text = text.lower()

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove headers / metadata-like lines
    text = re.sub(
        r"^\s*(from|subject|organization|lines|reply-to|writes):.*$",
        " ",
        text,
        flags=re.MULTILINE
    )

    # Keep alphabetic characters
    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal
    tokens = [
        word
        for word in tokens
        if word.isalpha()
        and word not in stop_words
        and len(word) > 1
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)


print("\n" + "=" * 70)
print("PREPROCESSING")
print("=" * 70)

df_selected["processed_text"] = (
    df_selected["text"]
    .apply(preprocess)
)


print("\nOriginal document:")
print(
    df_selected["text"].iloc[0][:500]
)

print("\nProcessed document:")
print(
    df_selected["processed_text"].iloc[0][:500]
)


# ============================================================
# 3 & 4. COSINE SIMILARITY FROM SCRATCH
# ============================================================

def cosine_similarity_manual(a, b):

    a = np.asarray(a).flatten()
    b = np.asarray(b).flatten()

    dot_product = np.dot(a, b)

    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (
        norm_a * norm_b
    )


# ============================================================
# EXPERIMENT FUNCTION
# ============================================================

def run_experiment(
    ngram_range,
    name
):

    print("\n\n")
    print("=" * 70)
    print(name.upper())
    print("=" * 70)


    # --------------------------------------------------------
    # 2. TF-IDF
    # --------------------------------------------------------

    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        min_df=1
    )

    X = vectorizer.fit_transform(
        df_selected["processed_text"]
    )

    print("\nTF-IDF matrix shape:", X.shape)

    print(
        "Number of features:",
        X.shape[1]
    )


    # Convert to NumPy
    X_dense = X.toarray()

    n = X_dense.shape[0]


    # ========================================================
    # 3. MANUAL COSINE SIMILARITY
    # ========================================================

    doc1 = 0
    doc2 = 1

    manual_similarity = cosine_similarity_manual(
        X_dense[doc1],
        X_dense[doc2]
    )

    print("\n" + "-" * 70)
    print("MANUAL COSINE SIMILARITY")
    print("-" * 70)

    print(
        "Document 1 class:",
        df_selected["class"].iloc[doc1]
    )

    print(
        "Document 2 class:",
        df_selected["class"].iloc[doc2]
    )

    print(
        "Cosine similarity:",
        round(manual_similarity, 6)
    )


    # ========================================================
    # 5. SIMILARITY MATRIX
    # ========================================================

    similarity_matrix = np.zeros(
        (n, n)
    )

    for i in range(n):

        for j in range(i, n):

            sim = cosine_similarity_manual(
                X_dense[i],
                X_dense[j]
            )

            similarity_matrix[i, j] = sim
            similarity_matrix[j, i] = sim


    # Self similarity should not be considered
    np.fill_diagonal(
        similarity_matrix,
        -1
    )


    # ========================================================
    # 5. TOP 5 SIMILAR DOCUMENTS
    # ========================================================

    top5 = []

    print("\n" + "-" * 70)
    print("TOP 5 SIMILAR DOCUMENTS")
    print("-" * 70)

    # Display for first 5 documents
    for i in range(5):

        indices = np.argsort(
            similarity_matrix[i]
        )[-5:][::-1]

        print(
            f"\nDocument {i} "
            f"({df_selected['class'].iloc[i]})"
        )

        for rank, j in enumerate(
            indices,
            start=1
        ):

            similarity = (
                similarity_matrix[i, j]
            )

            print(
                f"{rank}. "
                f"Document {j} | "
                f"Similarity = {similarity:.4f} | "
                f"Class = {df_selected['class'].iloc[j]}"
            )

            top5.append({
                "Document": i,
                "Similar Document": j,
                "Similarity": similarity,
                "Document Class":
                    df_selected["class"].iloc[i],
                "Similar Document Class":
                    df_selected["class"].iloc[j]
            })


    # ========================================================
    # 6. SAME CLASS VS DIFFERENT CLASS
    # ========================================================

    same_class = []
    different_class = []

    for i in range(n):

        for j in range(i + 1, n):

            similarity = similarity_matrix[i, j]

            if (
                df_selected["class"].iloc[i]
                ==
                df_selected["class"].iloc[j]
            ):

                same_class.append(
                    similarity
                )

            else:

                different_class.append(
                    similarity
                )


    average_same = np.mean(
        same_class
    )

    average_different = np.mean(
        different_class
    )


    print("\n" + "-" * 70)
    print("AVERAGE SIMILARITY")
    print("-" * 70)

    print(
        "Same-class similarity     :",
        round(average_same, 6)
    )

    print(
        "Different-class similarity:",
        round(average_different, 6)
    )

    separation = (
        average_same
        - average_different
    )

    print(
        "Class separation          :",
        round(separation, 6)
    )


    # ========================================================
    # 9. FIVE DOCUMENT PAIRS
    # ========================================================

    pairs = []

    for i in range(n):

        for j in range(i + 1, n):

            pairs.append({
                "Document 1": i,
                "Document 2": j,
                "Cosine Similarity":
                    similarity_matrix[i, j],
                "Class 1":
                    df_selected["class"].iloc[i],
                "Class 2":
                    df_selected["class"].iloc[j]
            })


    pair_df = pd.DataFrame(pairs)

    top_pairs = (
        pair_df
        .sort_values(
            "Cosine Similarity",
            ascending=False
        )
        .head(5)
    )


    print("\n" + "-" * 70)
    print("5 HIGHEST-SIMILARITY DOCUMENT PAIRS")
    print("-" * 70)

    print(
        top_pairs.to_string(
            index=False
        )
    )


    return {
        "name": name,
        "features": X.shape[1],
        "shape": X.shape,
        "average_same": average_same,
        "average_different":
            average_different,
        "separation": separation,
        "similarity_matrix":
            similarity_matrix,
        "top5": pd.DataFrame(top5),
        "pairs": top_pairs
    }


# ============================================================
# 7. UNIGRAM TF-IDF
# ============================================================

unigram_results = run_experiment(
    (1, 1),
    "TF-IDF Unigrams"
)


# ============================================================
# 7. UNIGRAM + BIGRAM TF-IDF
# ============================================================

bigram_results = run_experiment(
    (1, 2),
    "TF-IDF Unigrams + Bigrams"
)


# ============================================================
# 8 & 9. FINAL COMPARISON TABLE
# ============================================================

print("\n\n")
print("=" * 70)
print("FINAL COMPARISON TABLE")
print("=" * 70)


comparison = pd.DataFrame({

    "Representation": [
        "TF-IDF Unigrams",
        "TF-IDF Unigrams + Bigrams"
    ],

    "Documents": [
        unigram_results["shape"][0],
        bigram_results["shape"][0]
    ],

    "Features": [
        unigram_results["features"],
        bigram_results["features"]
    ],

    "Average Same-Class": [
        unigram_results["average_same"],
        bigram_results["average_same"]
    ],

    "Average Different-Class": [
        unigram_results["average_different"],
        bigram_results["average_different"]
    ],

    "Class Separation": [
        unigram_results["separation"],
        bigram_results["separation"]
    ]
})


print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# BIGRAM CONCLUSION
# ============================================================

print("\n" + "=" * 70)
print("BIGRAM ANALYSIS")
print("=" * 70)


if (
    bigram_results["separation"]
    >
    unigram_results["separation"]
):

    print(
        "Adding bigrams IMPROVED same-class "
        "identification because class separation "
        "increased."
    )

elif (
    bigram_results["separation"]
    <
    unigram_results["separation"]
):

    print(
        "Adding bigrams DID NOT IMPROVE same-class "
        "identification because class separation "
        "decreased."
    )

else:

    print(
        "Adding bigrams produced the same "
        "class separation as unigrams."
    )


# ============================================================
# FINAL 5 PAIRS
# ============================================================

print("\n" + "=" * 70)
print("5 PAIRS - UNIGRAM")
print("=" * 70)

print(
    unigram_results["pairs"].to_string(
        index=False
    )
)


print("\n" + "=" * 70)
print("5 PAIRS - UNIGRAM + BIGRAM")
print("=" * 70)

print(
    bigram_results["pairs"].to_string(
        index=False
    )
)

20 NEWSGROUPS DATASET

list.csv columns:
['newsgroup', 'document_id']

First rows:
            newsgroup  document_id
0  talk.religion.misc        82757
1  talk.religion.misc        82758
2  talk.religion.misc        82759
3  talk.religion.misc        82760
4  talk.religion.misc        82763

Number of category files: 20

Available categories:
misc.forsale
rec.autos
comp.os.ms-windows.misc
sci.electronics
comp.sys.mac.hardware
talk.politics.mideast
talk.politics.guns
talk.religion.misc
comp.graphics
soc.religion.christian
rec.sport.hockey
rec.sport.baseball
comp.windows.x
comp.sys.ibm.pc.hardware
rec.motorcycles
sci.med
sci.space
alt.atheism
sci.crypt
talk.politics.misc
misc.forsale                        -> 2041 documents
rec.autos                           -> 1574 documents
comp.os.ms-windows.misc             -> 1544 documents
sci.electronics                     -> 1324 documents
comp.sys.mac.hardware               -> 1278 documents
talk.politics.mideast               -> 2525 documen